# VoiceForge — speech backend (Colab **or** Kaggle)

Runs Chatterbox Multilingual on a free T4 and **registers itself** with your VoiceForge app. You never copy a tunnel URL again — press Run All and the backend appears as *Online* in Admin → Backends.

**Before running**
- **Colab:** Runtime → Change runtime type → **T4 GPU**
- **Kaggle:** Session options → Accelerator → **GPU T4 x2**, and **Internet → On**

Then fill in Cell 0 and **Run All**, top to bottom.

> Cell 1 prints a wall of red `pip` dependency-conflict lines about jax, opencv, cupy and friends. **Expected and harmless** — that's the platform's preinstalled data-science stack, which this backend never imports.

In [ ]:
# Cell 0 — your app details  (copy these from Admin → Backends)

VOICEFORGE_APP_URL = ""   # e.g. https://audio.bistab.com.np
REGISTRATION_TOKEN = ""   # the REGISTRATION_TOKEN you set in Vercel
BACKEND_SECRET     = ""   # the BACKEND_SECRET you set in Vercel

# Kaggle users: instead of pasting secrets here, you can put them in
# Add-ons → Secrets and leave the three lines above empty.
try:
    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
    VOICEFORGE_APP_URL = VOICEFORGE_APP_URL or _s.get_secret("VOICEFORGE_APP_URL")
    REGISTRATION_TOKEN = REGISTRATION_TOKEN or _s.get_secret("REGISTRATION_TOKEN")
    BACKEND_SECRET     = BACKEND_SECRET     or _s.get_secret("BACKEND_SECRET")
    print("Loaded settings from Kaggle Secrets.")
except Exception:
    pass

import os
# Which platform is this? A /kaggle directory now exists on Colab too, so
# that test mislabels a Colab session as Kaggle — and both would then fight
# over the same row in the registry. google.colab only imports on Colab, and
# KAGGLE_KERNEL_RUN_TYPE is only set by Kaggle, so ask those instead.
def _detect_provider() -> str:
    try:
        import google.colab  # noqa: F401
        return "colab"
    except ImportError:
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or os.environ.get("KAGGLE_URL_BASE"):
        return "kaggle"
    return "custom"

PROVIDER = _detect_provider()
os.environ["BACKEND_SECRET"] = BACKEND_SECRET
os.environ["VOICEFORGE_VOICES_DIR"] = "/tmp/voiceforge/voices"

assert VOICEFORGE_APP_URL, "Set VOICEFORGE_APP_URL above (Admin -> Backends shows it)."
assert REGISTRATION_TOKEN, "Set REGISTRATION_TOKEN above."
print(f"Provider: {PROVIDER}  ->  {VOICEFORGE_APP_URL}")

In [ ]:
# Cell 1 — Install
# numpy is the one thing that has to be right, and the window is narrow:
#   chatterbox-tts  numpy >= 2.0   (on Python 3.13; < 2.0 on 3.12)
#   numba (librosa) numpy <  2.3   -> "Numba needs NumPy 2.2 or less"
#   scipy           built against numpy 2.x on 3.13
# So the rule is a RANGE, not one version, and anything already inside it is
# left completely alone. Reinstalling a working numpy on top of Colab's image
# is what produced mixed installs and the 'numpy.ufunc has no attribute
# __module__' crash — the Python files and the compiled core disagreeing.
import subprocess, sys

# Nothing below works without network access, and the failures it causes are
# misleading: pip retries for two minutes then reports "no matching
# distribution", wget writes an empty file, and the download check blames the
# repo. Catch it here instead, in one second, with the actual reason.
import socket

try:
    socket.gethostbyname("pypi.org")
except OSError:
    raise SystemExit(
        "\n*** NO INTERNET IN THIS SESSION ***\n\n"
        "Kaggle: right sidebar > Session options > Internet > On.\n"
        "        Kaggle only offers that toggle on a phone-verified account\n"
        "        (kaggle.com > Settings > Phone Verification).\n"
        "Colab:  check the runtime is actually connected (top right).\n"
    )

PY313 = sys.version_info >= (3, 13)
NUMPY_SPEC = "numpy>=2.0,<2.3" if PY313 else "numpy>=1.24,<2.0"
NUMPY_PICK = "numpy==2.2.6" if PY313 else "numpy==1.26.4"


def run(args):
    return subprocess.run([sys.executable, "-m", "pip", *args], capture_output=True, text=True)


def installed(pkg):
    for line in run(["show", pkg]).stdout.splitlines():
        if line.startswith("Version:"):
            return line.split(":", 1)[1].strip()
    return None


def version_ok(v):
    if not v:
        return False
    major, minor = (int(x) for x in v.split(".")[:2])
    return (2, 0) <= (major, minor) < (2, 3) if PY313 else major == 1


def numpy_healthy():
    """A correct version number isn't enough — a half-replaced numpy reports
    the right version and still explodes on import. Prove it actually works,
    in a fresh interpreter so this cell's own imports can't mask it."""
    probe = subprocess.run(
        [sys.executable, "-c", "import numpy, numpy.strings; import scipy.special; print(numpy.__version__)"],
        capture_output=True, text=True,
    )
    return probe.returncode == 0, (probe.stderr or "").strip()[-500:]


print(f"Python {sys.version_info.major}.{sys.version_info.minor} -> needs {NUMPY_SPEC}")

!pip install -q chatterbox-tts fastapi uvicorn nest-asyncio python-multipart soundfile
!pip install -q torchvision==0.21.0

_v = installed("numpy")
_ok, _err = numpy_healthy()

if version_ok(_v) and _ok:
    print(f"numpy {_v} is in range and imports cleanly — leaving it alone.")
else:
    why = f"version {_v} out of range" if not version_ok(_v) else "installation is broken"
    print(f"numpy needs fixing ({why}); doing a clean reinstall of {NUMPY_PICK} …")
    if _err:
        print("  reported:", _err.splitlines()[-1] if _err.splitlines() else _err)
    # Uninstall FIRST. Installing over the top is what leaves a mix of files
    # from two versions behind, which is the state that crashes on import.
    !pip uninstall -q -y numpy
    !pip install -q --no-deps "{NUMPY_PICK}"

# cloudflared gives us a public HTTPS URL for this session.
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

# The server itself lives in the repo (backend/voiceforge_server.py) so Colab,
# Kaggle and Modal all run identical code. Download the current version.
!mkdir -p /content/backend && touch /content/backend/__init__.py
!wget -q -O /content/backend/voiceforge_server.py https://raw.githubusercontent.com/waytobijay/Bbox-Audio/main/backend/voiceforge_server.py

import os
_path = "/content/backend/voiceforge_server.py"
_size = os.path.getsize(_path) if os.path.exists(_path) else 0
print("\nshared server downloaded: %.1f KB" % (_size / 1000))
assert _size > 4000, (
    "Download failed. Either this session has no internet (Kaggle: Session "
    "options > Internet > On), or the repo is not public. You can also upload "
    "backend/voiceforge_server.py into /content/backend/ by hand."
)

# Prove the environment works HERE, before the restart, so Cell 2 can't be the
# thing that discovers a broken install.
_v = installed("numpy")
_ok, _err = numpy_healthy()
assert version_ok(_v) and _ok, (
    f"numpy is still wrong ({_v}).\n{_err}\n\n"
    "This runtime's packages are in a mixed state. Fix it the reliable way:\n"
    "  Runtime > Disconnect and delete runtime, then Run all on a fresh one."
)
print(f"numpy {_v}: imports cleanly with scipy  ✓")
print("\nInstalled. Now: Runtime > Restart session, then run Cell 0 and Cell 2.")

In [ ]:
# Cell 2 — Load the model
# Run this AFTER Runtime > Restart session, so the kernel picks up what Cell 1
# installed instead of the numpy it imported at startup.
import sys, torch
sys.path.insert(0, "/content")

assert torch.cuda.is_available(), (
    "No GPU. Colab: Runtime > Change runtime type > T4 GPU. "
    "Kaggle: Session options > Accelerator > GPU.")

import numpy as np
print(torch.cuda.get_device_name(0), "| torch", torch.__version__,
      "| numpy", np.__version__,
      "| py", f"{sys.version_info.major}.{sys.version_info.minor}")

# A RANGE, not one version — see Cell 1. Pinning an exact numpy here is what
# used to reject environments that were actually fine.
_major, _minor = (int(x) for x in np.__version__.split(".")[:2])
if sys.version_info >= (3, 13):
    _ok, _want = (_major, _minor) < (2, 3) and _major >= 2, ">=2.0,<2.3"
else:
    _ok, _want = _major == 1, ">=1.24,<2.0"
assert _ok, (
    f"numpy {np.__version__} is outside {_want} — re-run Cell 1, then "
    "Runtime > Restart session, then Cell 0 and Cell 2.")

# These two are where a mismatched numpy actually bites, deep inside the model
# load. Import them here so the failure is readable and instant.
import scipy.special
import numba

from backend.voiceforge_server import load_models, create_app, gpu_name, VERSION
MODEL_NAMES = load_models()
print("models ready:", MODEL_NAMES)

In [ ]:
# Cell 3 — Serve, open a tunnel, and REGISTER with your app
import json, re, subprocess, threading, time, urllib.request
import nest_asyncio, uvicorn

app = create_app(provider=PROVIDER)

nest_asyncio.apply()
threading.Thread(
    target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000, log_level="error"),
    daemon=True,
).start()
time.sleep(3)

# --- public URL ---
proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
PUBLIC_URL = None
for line in proc.stdout:
    m = re.search(r"https://[-\w]+\.trycloudflare\.com", line)
    if m:
        PUBLIC_URL = m.group(0)
        break
assert PUBLIC_URL, "Tunnel failed to start — re-run this cell."
print("tunnel:", PUBLIC_URL)

# --- register + heartbeat ---
REGISTER_URL = VOICEFORGE_APP_URL.rstrip("/") + "/api/backends/register"

def register():
    payload = json.dumps({
        "provider": PROVIDER,
        "url": PUBLIC_URL,
        "gpu": gpu_name(),
        "models": MODEL_NAMES,
        "version": VERSION,
    }).encode()
    req = urllib.request.Request(
        REGISTER_URL, data=payload, method="POST",
        headers={"Content-Type": "application/json",
                 "X-Registration-Token": REGISTRATION_TOKEN},
    )
    with urllib.request.urlopen(req, timeout=30) as r:
        return json.loads(r.read().decode())

try:
    result = register()
    print("\n" + "=" * 62)
    print("  REGISTERED — nothing to copy. Check Admin -> Backends.")
    print(f"  provider: {PROVIDER}   heartbeat: every {result.get('heartbeat_seconds', 60)}s")
    print("=" * 62 + "\n")
except Exception as e:
    print("\n*** REGISTRATION FAILED:", e)
    print("*** Check VOICEFORGE_APP_URL and REGISTRATION_TOKEN in Cell 0.")
    print("*** The backend is still running — you can add this URL by hand")
    print("*** under Admin -> Backends -> Custom:")
    print("   ", PUBLIC_URL)

# Heartbeats keep it showing Online; 3 missed beats marks it Offline.
def heartbeat_loop():
    while True:
        time.sleep(60)
        try:
            register()
        except Exception as e:
            print("heartbeat failed:", e, flush=True)

threading.Thread(target=heartbeat_loop, daemon=True).start()
print("Heartbeat running. Keep this tab open while you generate.")

In [ ]:
# Cell 4 — Keep-alive (optional)
# Colab/Kaggle disconnect idle tabs. Run this during long jobs.
import time
while True:
    time.sleep(60)
    print(".", end="", flush=True)